# 🏛️ oracc-parser — Quickstart

This notebook shows you the actual data available, how to parse it, and where everything lives on disk.

In [ ]:
!pip install display

## 🎯 Goals of this notebook

1. **Confirm data is available** — check that downloaded project ZIPs are present
2. **Understand the directory layout** — data, cache, output, jsonzip directories
3. **Browse available projects** — see what's downloaded locally vs. what exists on ORACC
4. **Parse a project** — run the full pipeline on a real ORACC corpus
5. **Explore tablets** — inspect transliterations, metadata, and sign-level Unicode cuneiform
6. **See the reference data** — provenance, periods, sign list

## 1. Get the data (Zenodo)

First, let's make sure you have the pre-downloaded data (project ZIPs, translations, Pleiades).
This downloads ~770MB from Zenodo so you don't have to scrape ORACC yourself.

In [ ]:
import os
from pathlib import Path
from oracc_parser.settings import DATA_DIR, OUTPUT_DIR, CACHE_DIR, JSONZIP_DIR

# Check if data exists
zip_dir = JSONZIP_DIR
if not zip_dir.exists() or len(list(zip_dir.glob("*.zip"))) == 0:
    print("⬇️  Data missing. Downloading from Zenodo... (this may take a minute)")
    # Run the download script (works in Jupyter)
    %run ../scripts/download_zenodo_data.py
else:
    print(f"✅ Data found in {zip_dir} ({len(list(zip_dir.glob('*.zip')))} projects)")

## 2. What data do we have?

Let's look at what's already downloaded and what's available.

In [ ]:
from pathlib import Path
from oracc_parser import reference_data


# What directories does oracc-parser use?
print("📁 Directory layout:")
print(f"   Data dir:    {DATA_DIR}")
print(f"   JSON ZIPs:   {JSONZIP_DIR}")
print(f"   Output dir:  {OUTPUT_DIR}")
print(f"   Cache dir:   {CACHE_DIR}")

In [ ]:
# What ORACC projects exist in the world?
# We fetch the LIVE list from ORACC servers:
all_projects = reference_data.get_live_project_list()
print(f"🌍 ORACC has {len(all_projects)} public projects")
all_projects.head(10)

In [ ]:
# Which ones have we already downloaded?
import os
import csv
from oracc_parser.settings import jsonzip_dir
from oracc_parser.utils.paths import get_projects_metadata

# Load projects with empty text folders so we can exclude them
_meta = get_projects_metadata()
_empty_projects = set(
    _meta.loc[_meta['Is_Text_Folder_Empty'].str.lower() == 'yes', 'Project_Name']
)

# Check for downloaded ZIPs using configured path
zip_dir = jsonzip_dir()
print(zip_dir)
if zip_dir.exists():
    # Exclude the large Zenodo bundle ZIPs and projects with no texts
    _zenodo_zips = {'oracc_jsonzip_all.zip', 'oracc_html_translations.zip', 'plaides_scraped_data.zip'}
    project_zips = [
        f for f in zip_dir.glob('*.zip')
        if f.name not in _zenodo_zips
        and f.stem.replace('-', '/') not in _empty_projects
    ]
    downloaded = sorted([f.stem for f in project_zips])
    total_size_mb = sum(f.stat().st_size for f in project_zips) / (1024*1024)
    print(f"📦 {len(downloaded)} projects downloaded ({total_size_mb:.0f} MB total)")
    print()
    # Show first 20
    for i, name in enumerate(downloaded[:20]):
        size_mb = (zip_dir / f"{name}.zip").stat().st_size / (1024*1024)
        print(f"   {name:40s}  {size_mb:6.1f} MB")
    if len(downloaded) > 20:
        print(f"   ... and {len(downloaded)-20} more")
else:
    print("No downloaded projects found yet.")
    print("Run: python scripts/download_zenodo_data.py")


In [ ]:
# Which ones have we already downloaded?
from oracc_parser.utils.paths import get_projects_metadata

# Load projects with empty text folders so we can exclude them
_meta = get_projects_metadata()
_empty_projects = set(
    _meta.loc[_meta['Is_Text_Folder_Empty'].str.lower() == 'yes', 'Project_Name']
)

# Check for downloaded ZIPs using configured path
zip_dir = jsonzip_dir()
print(zip_dir)
if zip_dir.exists():
    # Exclude the large Zenodo bundle ZIPs and projects with no texts
    _zenodo_zips = {'oracc_jsonzip_all.zip', 'oracc_html_translations.zip', 'plaides_scraped_data.zip'}
    project_zips = [
        f for f in zip_dir.glob('*.zip')
        if f.name not in _zenodo_zips
        and f.stem.replace('-', '/') not in _empty_projects
    ]
    downloaded = sorted([f.stem for f in project_zips])
    total_size_mb = sum(f.stat().st_size for f in project_zips) / (1024*1024)
    print(f"📦 {len(downloaded)} projects downloaded ({total_size_mb:.0f} MB total)")
    print()
    # Show first 20
    for i, name in enumerate(downloaded[:20]):
        size_mb = (zip_dir / f"{name}.zip").stat().st_size / (1024*1024)
        print(f"   {name:40s}  {size_mb:6.1f} MB")
    if len(downloaded) > 20:
        print(f"   ... and {len(downloaded)-20} more")
else:
    print("No downloaded projects found yet.")
    print("Run: python scripts/download_zenodo_data.py")


In [ ]:
# Compare: which projects are NOT yet downloaded?
from oracc_parser.utils.paths import get_projects_metadata
# Load projects with empty text folders so we can exclude them
_meta = get_projects_metadata()
_empty_projects = set(
    _meta.loc[_meta['Is_Text_Folder_Empty'].str.lower() == 'yes', 'Project_Name']
)
# Check for downloaded ZIPs using configured path
zip_dir = jsonzip_dir()

if zip_dir.exists() and 'all_projects' in dir():
    _zenodo_zips = {'oracc_jsonzip_all.zip', 'oracc_html_translations.zip', 'plaides_scraped_data.zip'}

    downloaded_names = {
        f.stem.replace('-', '/') for f in zip_dir.glob('*.zip')
        if f.name not in _zenodo_zips
        and f.stem.replace('-', '/') not in _empty_projects
    }
    live_names = set(all_projects['project'].tolist()) if 'project' in all_projects.columns else set()

    # Exclude empty-text projects from the live list too
    live_names -= _empty_projects

    not_downloaded = sorted(live_names - downloaded_names)

    print(f"📊 Download status:")
    print(f"   ✅ {len(downloaded_names)} projects downloaded")
    print(f"   ❌ {len(not_downloaded)} projects NOT yet downloaded")
    print(f"   🌍 {len(live_names)} total projects on ORACC (with texts)")
    print()
    if not_downloaded:
        print("Projects not yet downloaded:")
        for name in not_downloaded:
            print(f"   • {name}")
        print()
        print("To download one: oracc-parser download --project <name>")
        if len(not_downloaded) > 10:
            print("NOTE: If there are many new projects or a large mismatch, it means ORACC has updated significant projects.")
            print("   We recommend downloading the new projects directly using the function below.")
    else:
        print("🎉 All available projects are downloaded!")


In [ ]:
# ⬇️ Download Data
# You can download any project that was not yet downloaded by us, by name.
from oracc_parser.download.oracc_download import download_projects

projects_to_download = [
   "saao/saa01"
]


if projects_to_download:
    download_projects(projects_to_download)


## 3. Parse a project

Pick any project from the lists above. We'll parse a small sample to see the output.

In [ ]:
from oracc_parser import parse_project, RunConfig
from oracc_parser.settings import JSONZIP_DIR

# Change this to any project you want!
PROJECT = "saao/saa01"  
LIMIT = 5  # Set to None to parse everything

# Check that the project is available locally
zip_dir = JSONZIP_DIR
zip_name = PROJECT.replace('/', '-') + '.zip'
if not (zip_dir / zip_name).exists():
    available = sorted(f.stem.replace('-', '/') for f in zip_dir.glob('*.zip'))
    print(f"⚠️  Project '{PROJECT}' is not downloaded!")
    print(f"   Available projects ({len(available)} total): {available[:10]}...")
    print(f"   To download: oracc-parser download --project {PROJECT}")
else:
    config = RunConfig(limit=LIMIT)
    records = parse_project(PROJECT, config=config)
    print(f"✓ Parsed {len(records)} tablets from {PROJECT}")

> **💡 Caching:** The first `parse_project()` call parses everything (slow).
> Subsequent calls with the **same config** return instantly from cache.
> Switching configs (e.g. `mask_pos`, `drop_missing`) reuses the cached words
> and only rebuilds the string representations — still much faster than re-parsing.
> See **Notebook 03** for details, or use `RunConfig(USE_CACHE=False)` to force re-parsing.

In [ ]:
# Look at one tablet up close
tablet = records[0]

print(f"=== {tablet.metadata.identifier} ===")
print(f"   Project:     {PROJECT}")
print(f"   Text ID:     {tablet.metadata.id_text}")
print(f"   Genre:       {tablet.metadata.genre}")

geo = tablet.metadata.geographical_information
if geo:
    print(f"   Provenance:  {geo.city.city_name}")
    print(f"   Pleiades ID: {geo.city.city_plaides_id}")

chrono = tablet.metadata.chronological_information
if chrono:
    print(f"   Period:      {chrono.tablet_period.period_name if chrono.tablet_period else ''}")
    print(f"   Years:       {chrono.start_year} to {chrono.end_year}")

print(f"   Words:       {len(tablet.content.words)}")

In [ ]:
# The text in different representations
c = tablet.content

print("TRANSLITERATION:")
print(f"{c.transliterated_str_representation.text[:200] if c.transliterated_str_representation else '(none)'}")
print()
print("NORMALIZATION:")
print(f"{c.normalized_str_representation.text[:200] if c.normalized_str_representation else '(none)'}")
print()
print("UNICODE CUNEIFORM:")
print(f"{c.unicode_str_representation.text[:200] if c.unicode_str_representation else '(none)'}")
print()
print("ENGLISH TRANSLATION:")
print(f"{c.english_translation[:200] if c.english_translation else '(none)'}")

## 5. Get flat DataFrames (for analysis & export)

Each function returns a clean pandas DataFrame — no nesting, no Pydantic.

In [ ]:
from oracc_parser import get_metadata_table, get_transliterations, get_full_flat_table

# Metadata table
print("=== METADATA ===")
meta = get_metadata_table(records)
display(meta)

In [ ]:
# Transliterations
print("=== TRANSLITERATIONS ===")
trans = get_transliterations(records)
display(trans)

In [ ]:
# Full flat table — everything in one DataFrame
flat = get_full_flat_table(records)
print(f"Full table: {flat.shape[0]} rows × {flat.shape[1]} columns")
print(f"Columns: {list(flat.columns)}")
display(flat)

## 6. Export & see where output goes

In [ ]:
# Export as JSONL
out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

jsonl_path = out_dir / f"{PROJECT.replace('/', '_')}.jsonl"
csv_path = out_dir / f"{PROJECT.replace('/', '_')}.csv"

flat.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
flat.to_csv(csv_path, index=False)

print(f"✓ Exported to:")
print(f"   JSONL: {jsonl_path}  ({jsonl_path.stat().st_size / 1024:.1f} KB)")
print(f"   CSV:   {csv_path}  ({csv_path.stat().st_size / 1024:.1f} KB)")
print()
print(f"📁 Output directory: {out_dir}")
for f in sorted(out_dir.iterdir()):
    print(f"   {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

## 7. Reference data (bundled with the package)

These datasets ship with oracc-parser — no download needed.

In [ ]:
from oracc_parser import reference_data

print("Bundled reference data:")
datasets = {
    "Provenance (cities → Pleiades IDs)": reference_data.get_provenance(),
    "Period mapping (period → years)": reference_data.get_period_mapping(),
    "Sign readings (8900+ signs)": reference_data.get_sign_list(),
    "POS tags": reference_data.get_pos_tags(),
    "Languages": reference_data.get_languages(),
    "ORACC projects metadata": reference_data.get_projects_metadata(),
}
for name, df in datasets.items():
    print(f"   {name:45s}  {len(df):>6} rows × {len(df.columns)} cols")

In [ ]:
# Explore any of them
print("=== Provenance ===")
display(reference_data.get_provenance().head(10))

print("\n=== Period mapping ===")
display(reference_data.get_period_mapping())

## What's next?

- **Clean the data:** See notebook `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **Explore reference data:** See notebook `02_reference_data.ipynb` for a catalog of available projects, pos tags, and languages.